# 01 — Know Your Data Before You Model It
## Loading, Understanding, and Preprocessing Human iEEG

---

> *"A model trained on poorly understood data is a sophisticated way of fitting noise."*

The single most common mistake in computational neuroscience is building a complex model before deeply understanding the raw signal. This notebook runs before any model. Its output is a clean, well-characterized dataset and a set of questions you can actually answer.

**What you will understand after this notebook:**
1. What ECoG actually measures — the physics of the signal
2. The full preprocessing pipeline and *why* each step exists
3. The Miller N-back task design — what happened in the scanner, sample by sample
4. High-gamma power: what it is, why it tracks cognition, how to extract it
5. The key comparison: neural activity under different N-back loads and target detection

**Data summary (from inspection):**
- Subject `al`: 40 channels, 360120 samples, 1200 Hz, 3 conditions (0/1/2-back)
- Subjects `ca`, `cc`, `ug`: 62-64 channels, ~980000 samples, includes rest block
- `stim`: letter identity (1-40 = letter shown, 0 = inter-stimulus interval)
- `task`: condition (-1=rest, 0=0-back, 1=1-back, 2=2-back)
- `target`: trial type (0=no stimulus, 1=non-target, 2=target/match)
- Data is already bandpass filtered 1-300 Hz by Miller lab (see `nsfilt`)

In [ ]:
import numpy as np
import scipy.signal as sig
import scipy.io as sio
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
import sys

# Add scripts directory to path
sys.path.insert(0, str(Path('../../scripts').resolve()))
from ieeg_utils import (
    DATA_DIR, SRATE, TASK_CODES, TARGET_CODES,
    load_subject, preprocess, high_gamma_power, epoch_data, baseline_normalize,
    reject_bad_channels, pca_project
)

plt.rcParams.update({'font.size': 11, 'figure.dpi': 110})
print('Data directory:', DATA_DIR)
print('Files:', list(DATA_DIR.glob('*.mat')))

---
## 1. What Are We Measuring? The Physics of ECoG

**Electrocorticography (ECoG)** = electrodes placed directly on the cortical surface, recording the **local field potential (LFP)**: the summed electrical activity of thousands of neurons within ~1-2 mm of each electrode.

**What generates the signal:**
- Primarily: **postsynaptic currents** — when a synapse releases neurotransmitter, ions flow across the membrane, generating a current dipole. Thousands of aligned pyramidal neurons (aligned perpendicular to cortex) create a detectable extracellular field.
- Secondarily: **afterhyperpolarizations**, **gap junctions**, axonal currents (weaker)

**The frequency hierarchy:**
| Band | Frequency | Source | Cognitive correlate |
|------|-----------|--------|--------------------|
| Delta | 1-4 Hz | Large-scale synchrony | Sleep, disorders |
| Theta | 4-8 Hz | Hippocampal-cortical | WM encoding, navigation |
| Alpha | 8-12 Hz | Thalamic pacing | Attention, inhibition |
| Beta | 13-30 Hz | Motor cortex, PFC | WM maintenance |
| **High-gamma** | **70-150 Hz** | **Local MUA** | **Active processing** |

**Why high-gamma for WM:** High-gamma is **broadband** (it's not an oscillation — it's elevated noise floor). It correlates with multi-unit activity (MUA) — the aggregate firing of neurons near the electrode. It tracks cognitive load, attention, and WM maintenance with high spatial and temporal precision.

**Important:** The Miller data is NOT raw voltage. It is already filtered 1-300 Hz by the recording system. The `nsfilt` array is the filter's frequency response.

In [ ]:
# ─── Load subject 'al' ────────────────────────────────────────────────────────
subj = 'al'
d    = load_subject(subj)

data   = d['data']       # (360120, 40) — raw ECoG, already 1-300 Hz filtered
stim   = d['stim']       # (360120,) — letter identity
task   = d['task']       # (360120,) — condition
target = d['target']     # (360120,) — trial type
srate  = d['srate']      # 1200

n_samples, n_channels = data.shape
duration_s = n_samples / srate

print(f"Subject: {subj}")
print(f"Duration: {duration_s:.1f} s = {duration_s/60:.1f} min")
print(f"Channels: {n_channels}")
print(f"Sampling rate: {srate} Hz")
print(f"Conditions: {[TASK_CODES[k] for k in sorted(np.unique(task))]}")
print()

# Block structure
for cond in np.unique(task):
    n = (task == cond).sum()
    print(f"  {TASK_CODES[cond]:10s}: {n} samples = {n/srate:.1f} s")

In [ ]:
# ─── Inspect the Miller lab's pre-applied filter ────────────────────────────
ns = sio.loadmat(str(DATA_DIR / 'ns_1k_1_300_filt.mat'))
nsfilt = ns['nsfilt'].ravel()  # frequency-domain magnitude response

# The filter has 300 values — one per Hz from 1 to 300
freqs = np.arange(1, 301)

plt.figure(figsize=(10, 3))
plt.plot(freqs, nsfilt, 'steelblue', lw=1.5)
plt.xlabel('Frequency (Hz)')
plt.ylabel('Filter magnitude')
plt.title('Miller Lab Pre-filter: 1-300 Hz bandpass magnitude response')
plt.axhline(1.0, color='k', lw=0.5, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print("Note: The data has already been bandpass filtered.")
print("We still need: notch (line noise) + CAR (common average reference)")

---
## 2. Look at the Raw Data

**Always look at your raw data before running any analysis.**

This is not optional. Real iEEG data contains:
- Broken electrodes (flat line or saturated)
- Movement artifacts (sudden amplitude jumps)
- Line noise (60 Hz and harmonics)
- Epileptic activity (the patients are epilepsy patients)
- Reference channel problems

You cannot model what you cannot see. Plot 5 random channels first.

In [ ]:
# ─── Plot raw traces — first 5 seconds of 0-back block ───────────────────────
t_start = 0
t_end   = int(5 * srate)   # 5 seconds
t_axis  = np.arange(t_start, t_end) / srate

n_show  = 8  # show 8 channels
ch_idxs = np.linspace(0, n_channels-1, n_show, dtype=int)

fig, axes = plt.subplots(n_show, 1, figsize=(14, 10), sharex=True)
for i, (ax, ch) in enumerate(zip(axes, ch_idxs)):
    trace = data[t_start:t_end, ch]
    ax.plot(t_axis, trace, 'k', lw=0.5)
    ax.set_ylabel(f'Ch {ch}', fontsize=9)
    ax.set_yticks([])
    # Shade stimulus periods
    stim_on = stim[t_start:t_end] > 0
    ax.fill_between(t_axis, trace.min(), trace.max(), where=stim_on,
                    alpha=0.15, color='gold', label='stimulus')

axes[-1].set_xlabel('Time (s)')
fig.suptitle(f'Raw ECoG — Subject {subj}, first 5 seconds (0-back block)', fontweight='bold')
plt.tight_layout()
plt.show()

# Check for obviously bad channels
ch_var  = data.var(axis=0)
print("Channel variance statistics:")
print(f"  Median: {np.median(ch_var):.4f}")
print(f"  Max:    {ch_var.max():.4f}  (channel {ch_var.argmax()})")
print(f"  Min:    {ch_var.min():.4f}  (channel {ch_var.argmin()})")

In [ ]:
# ─── Power spectral density: what frequencies are in the signal? ─────────────
# Use Welch's method: average periodogram over overlapping windows
# WHY Welch: single-channel FFT is noisy — Welch's method reduces variance by
#            averaging spectra from multiple windows.

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot PSD for 3 channels in each condition
cond_colors = {0: 'royalblue', 1: 'seagreen', 2: 'crimson'}

for cond in [0, 1, 2]:
    mask = task == cond
    ch   = 10  # one representative channel
    trace_cond = data[mask, ch]
    freqs_psd, psd = sig.welch(trace_cond, fs=srate, nperseg=2048)
    axes[0].semilogy(freqs_psd, psd, color=cond_colors[cond],
                     label=TASK_CODES[cond], lw=1.5)

axes[0].set_xlabel('Frequency (Hz)')
axes[0].set_ylabel('Power spectral density')
axes[0].set_title('PSD by condition — channel 10')
axes[0].axvspan(70, 150, alpha=0.12, color='gold', label='High-gamma')
axes[0].legend(fontsize=9)
axes[0].set_xlim(0, 300)

# Power map: mean power per channel (look for 60 Hz peaks = line noise)
all_psd = []
for ch in range(n_channels):
    f, p = sig.welch(data[:, ch], fs=srate, nperseg=2048)
    all_psd.append(p)
all_psd = np.array(all_psd)  # (C, freqs)

im = axes[1].imshow(np.log10(all_psd[:, :150]), aspect='auto',
                    origin='lower', cmap='viridis',
                    extent=[0, f[149], 0, n_channels])
plt.colorbar(im, ax=axes[1], label='log10 power')
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Channel')
axes[1].set_title('Power map across all channels')
# Mark line noise
for ln in [60, 120, 180]:
    axes[1].axvline(ln, color='white', lw=1, linestyle='--', alpha=0.7)

plt.suptitle('Power Spectral Analysis', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Preprocessing — Every Step Has a Reason

The pipeline:
1. **Channel rejection** — remove broken electrodes
2. **Common average reference (CAR)** — remove shared-mode noise
3. **Notch filter** — remove 60 Hz line noise and harmonics

The data is already 1-300 Hz bandpass filtered by the recording system, so we skip that step.

In [ ]:
# ─── Channel rejection ────────────────────────────────────────────────────────
good_channels = reject_bad_channels(data, threshold_sd=3.0)
print(f"Good channels: {good_channels.sum()} / {n_channels}")
print(f"Rejected: {np.where(~good_channels)[0]}")

data_good = data[:, good_channels]   # keep only good channels
n_good    = data_good.shape[1]

# ─── CAR + notch ──────────────────────────────────────────────────────────────
print("Applying CAR and notch filters...")
data_clean = preprocess(data_good, srate=srate)
print("Done.")

# Show effect of preprocessing on one channel
ch_show = 5
t_show  = slice(0, int(3 * srate))  # 3 seconds
t_ax    = np.arange(int(3 * srate)) / srate

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(t_ax, data_good[t_show, ch_show], 'k', lw=0.4)
axes[0].set_title('Before preprocessing'); axes[0].set_ylabel('Voltage (µV)')
axes[1].plot(t_ax, data_clean[t_show, ch_show], 'steelblue', lw=0.4)
axes[1].set_title('After CAR + notch'); axes[1].set_ylabel('Voltage (µV)')
axes[1].set_xlabel('Time (s)')
plt.suptitle(f'Preprocessing effect — channel {ch_show}', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. High-Gamma Power Extraction

**Why Hilbert transform:**
1. Bandpass to 70-150 Hz
2. Hilbert transform → analytic signal $z(t) = x(t) + i\hat{x}(t)$
3. Envelope: $|z(t)|$ = instantaneous amplitude
4. Power: $|z(t)|^2$
5. Smooth: Gaussian kernel (50 ms) — reduces noise, preserves task-related changes

**The result:** A signal that tells you "how much broadband cortical activity is happening right now at this electrode."

In [ ]:
# ─── Extract high-gamma power ─────────────────────────────────────────────────
print("Extracting high-gamma power (70-150 Hz)...")
hgp = high_gamma_power(data_clean, srate=srate, lo=70, hi=150, smooth_ms=50)
print(f"High-gamma power shape: {hgp.shape}")

# Show comparison: raw vs high-gamma power on one channel
t_show = slice(int(100 * srate), int(103 * srate))  # 3s of 0-back
t_ax   = np.arange(int(3 * srate)) / srate

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)

ch_show = 5
axes[0].plot(t_ax, data_clean[t_show, ch_show], 'k', lw=0.4)
axes[0].set_title('Preprocessed ECoG')

# Show the 70-150 Hz filtered signal
from ieeg_utils import bandpass_filter
hg_band = bandpass_filter(data_clean[:, ch_show], 70, 150, srate)
axes[1].plot(t_ax, hg_band[t_show], color='steelblue', lw=0.4)
axes[1].set_title('High-gamma bandpass (70-150 Hz)')

axes[2].plot(t_ax, hgp[t_show, ch_show], color='crimson', lw=1.5)
axes[2].set_title('High-gamma power (Hilbert envelope², smoothed 50 ms)')
axes[2].set_xlabel('Time (s)')

# Mark stimulus onsets
from ieeg_utils import find_stimulus_onsets
onsets_all = find_stimulus_onsets(stim)
onsets_win = onsets_all[(onsets_all >= t_show.start) & (onsets_all < t_show.stop)] - t_show.start
for onset in onsets_win:
    for ax in axes:
        ax.axvline(onset / srate, color='gold', lw=1.5, alpha=0.7)

[ax.set_ylabel('a.u.') for ax in axes]
plt.suptitle(f'Signal extraction pipeline — channel {ch_show}', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Epoching — Cutting Continuous Signal Into Trials

The continuous recording contains all trials mixed together. We **epoch** (cut) the signal into fixed windows around each stimulus onset.

Convention:
- **Pre-stimulus baseline**: -200 ms to 0 ms (before letter appears)
- **Post-stimulus window**: 0 ms to 1500 ms (encoding + early maintenance)

Result: a tensor of shape `(n_trials, n_times, n_channels)`

In [ ]:
# ─── Epoch the high-gamma power ───────────────────────────────────────────────
epochs_dict = epoch_data(hgp, stim, task, target,
                          pre_ms=200, post_ms=1500, srate=srate)

epochs = epochs_dict['epochs']    # (n_trials, n_times, n_channels)
times  = epochs_dict['times']     # (n_times,) in seconds
task_id = epochs_dict['task_id']  # (n_trials,) condition
tgt_id  = epochs_dict['tgt_id']  # (n_trials,) target type

print(f"Epochs shape: {epochs.shape}")
print(f"Time range: {times[0]:.2f} to {times[-1]:.2f} s")
print(f"Trials per condition:")
for cond in np.unique(task_id):
    n = (task_id == cond).sum()
    print(f"  {TASK_CODES[cond]:10s}: {n} trials")
print(f"\nTrials by target type:")
for ttype in np.unique(tgt_id):
    n = (tgt_id == ttype).sum()
    print(f"  {TARGET_CODES[ttype]:12s}: {n} trials")

# Baseline normalize
epochs_z = baseline_normalize(epochs, times, baseline_window=(-0.2, 0.0))
print("\nBaseline normalized: units now = SDs above pre-stimulus baseline")

---
## 6. The Key Comparison — Cognitive Load Effects

**N-back task logic:**
- 0-back: just press for a specific pre-defined letter — pure visual detection, no WM
- 1-back: press if current = 1 letter ago — minimal WM load
- 2-back: press if current = 2 letters ago — high WM load

**The prediction from WM neuroscience:** PFC high-gamma power should scale with load (0 < 1 < 2-back) during the maintenance window (500-1500 ms). This is the basic result this dataset should replicate — it was published in the Miller 2007/2016 papers.

If we can replicate the load effect in high-gamma, we can trust the data quality before building the geometry analysis.

In [ ]:
# ─── ERP (Event-Related Power) by condition ───────────────────────────────────
# Average high-gamma power across trials, per condition, per channel

cond_colors = {0: 'royalblue', 1: 'seagreen', 2: 'crimson'}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Top row: mean high-gamma timecourse for each condition (selected channels)
ch_examples = [0, 5, 15]  # 3 example channels
for col, ch in enumerate(ch_examples):
    ax = axes[0, col]
    for cond in [0, 1, 2]:
        mask   = task_id == cond
        mean_  = epochs_z[mask, :, ch].mean(axis=0)
        sem_   = epochs_z[mask, :, ch].std(axis=0) / np.sqrt(mask.sum())
        ax.plot(times, mean_, color=cond_colors[cond], lw=2, label=TASK_CODES[cond])
        ax.fill_between(times, mean_-sem_, mean_+sem_,
                        color=cond_colors[cond], alpha=0.15)
    ax.axhline(0, color='k', lw=0.5)
    ax.axvline(0, color='gray', lw=1, linestyle='--')
    ax.set_title(f'Channel {ch}')
    ax.set_xlabel('Time from stimulus onset (s)')
    if col == 0:
        ax.set_ylabel('High-gamma power (z-score)')
        ax.legend(fontsize=9)
    ax.axvspan(0.5, 1.5, alpha=0.08, color='gold')  # maintenance window

# Bottom row: target vs non-target
for col, cond in enumerate([0, 1, 2]):
    ax = axes[1, col]
    ch = 5
    for ttype, color, label in [(1, 'steelblue', 'Non-target'), (2, 'orangered', 'Target')]:
        mask = (task_id == cond) & (tgt_id == ttype)
        if mask.sum() < 3:
            continue
        mean_ = epochs_z[mask, :, ch].mean(axis=0)
        sem_  = epochs_z[mask, :, ch].std(axis=0) / np.sqrt(mask.sum())
        ax.plot(times, mean_, color=color, lw=2, label=f'{label} (n={mask.sum()})')
        ax.fill_between(times, mean_-sem_, mean_+sem_, color=color, alpha=0.15)
    ax.axhline(0, color='k', lw=0.5)
    ax.axvline(0, color='gray', lw=1, linestyle='--')
    ax.set_title(f'{TASK_CODES[cond]}: target vs non-target')
    ax.set_xlabel('Time from stimulus onset (s)')
    if col == 0:
        ax.set_ylabel('High-gamma power (z-score)')
    ax.legend(fontsize=9)

plt.suptitle(f'High-Gamma Power by Condition — Subject {subj}', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── Spatial map: which channels respond to WM load? ─────────────────────────
# For each channel, compute mean high-gamma in maintenance window (0.5-1.5s)
# for each condition. Show the difference: 2-back minus 0-back.

maint_mask = (times >= 0.5) & (times <= 1.5)

mean_by_cond = {}  # cond → (n_channels,)
for cond in [0, 1, 2]:
    mask    = task_id == cond
    # Mean over time (maintenance) and trials
    mean_by_cond[cond] = epochs_z[mask][:, maint_mask, :].mean(axis=(0, 1))

contrast = mean_by_cond[2] - mean_by_cond[0]  # 2-back minus 0-back

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart of mean activity per condition (all channels averaged)
cond_means = [mean_by_cond[c].mean() for c in [0, 1, 2]]
cond_sems  = [mean_by_cond[c].std() / np.sqrt(n_good) for c in [0, 1, 2]]
axes[0].bar([0, 1, 2], cond_means, yerr=cond_sems,
            color=['royalblue', 'seagreen', 'crimson'], edgecolor='white')
axes[0].set_xticks([0, 1, 2])
axes[0].set_xticklabels(['0-back\n(no memory)', '1-back\n(low load)', '2-back\n(high load)'])
axes[0].set_ylabel('Mean HGP (z-score, maintenance)')
axes[0].set_title('WM Load Effect (averaged over channels)')

# Per-channel contrast: 2-back minus 0-back
axes[1].bar(range(n_good), contrast,
            color=['crimson' if c > 0 else 'steelblue' for c in contrast])
axes[1].axhline(0, color='k', lw=0.5)
axes[1].set_xlabel('Channel index')
axes[1].set_ylabel('HGP contrast (2-back − 0-back)')
axes[1].set_title('Per-channel WM Load Sensitivity')

plt.suptitle('Working Memory Load Effect in High-Gamma Power', fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Mean HGP during maintenance:")
for cond in [0, 1, 2]:
    print(f"  {TASK_CODES[cond]:12s}: {mean_by_cond[cond].mean():.4f} ± {mean_by_cond[cond].std():.4f}")
print(f"\nLoad effect (2-back − 0-back): {contrast.mean():.4f}")

In [ ]:
# ─── Preview: PCA trajectories by condition ───────────────────────────────────
# This is a teaser for Module 2. Take all trials, flatten to (n_trials*n_times, n_ch),
# project to 3D, plot each trial as a trajectory in PCA space.

# Stack all epochs: (n_trials * n_times, n_channels)
n_trials, n_times_ep, n_ch = epochs_z.shape
X_all = epochs_z.reshape(-1, n_ch)

# PCA
scores, components, var_exp = pca_project(X_all, n_components=3)
scores_3d = scores.reshape(n_trials, n_times_ep, 3)   # (n_trials, n_times, 3)

print(f"Variance explained by top 3 PCs: {var_exp.sum()*100:.1f}%")

fig = plt.figure(figsize=(14, 5))
for col, cond in enumerate([0, 1, 2]):
    ax = fig.add_subplot(1, 3, col+1, projection='3d')
    mask = task_id == cond
    # Plot a sample of trials
    n_plot = min(20, mask.sum())
    trial_idxs = np.where(mask)[0][:n_plot]
    for ti in trial_idxs:
        traj = scores_3d[ti]   # (n_times, 3)
        ax.plot(traj[:,0], traj[:,1], traj[:,2],
                color=list(cond_colors.values())[cond], lw=0.8, alpha=0.5)
    ax.set_title(f'{TASK_CODES[cond]}\n({n_plot} trials shown)')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.set_zlabel('PC3')

plt.suptitle('Neural Trajectories in PCA Space — Teaser for Module 2', fontweight='bold')
plt.tight_layout()
plt.show()
print("In Module 2, we'll learn a MUCH better latent space via Sequential VAE.")

---
## ✏️ Exercises — Mastery Tests

These require thought, not just running code. Write your reasoning before implementing.

### Exercise A — Preprocessing Audit

1. Re-run the analysis WITHOUT the common average reference. How do the ERP plots change? Why? What does this tell you about the source of the signal you're measuring?

2. Re-run WITHOUT the notch filter. What artifacts appear in the high-gamma power? Why do 60 Hz and 120 Hz appear in the power even when the data is already bandpass filtered to 300 Hz?

3. Try threshold_sd = 1.5 for channel rejection (stricter) vs. 5.0 (looser). How many channels do you lose? Plot the per-channel variance to understand which channels are borderline.

### Exercise B — The Replicate

Run the full analysis on a second subject (`ca` or `cc`). Do you see the same WM load effect? If it's weaker for one subject, what are three possible explanations (signal quality, electrode placement, task compliance, sample size)?

### Exercise C — Design Question (important)

The target variable has values 1 (non-target) and 2 (target). But we don't have the subject's *button press* — we don't know if they got it right.

1. For the 2-back condition: what does a *missed target* (target trial with no response) look like in neural data, based on what you know about WM?

2. Can you infer behavioral performance from the neural data alone (without button press)? What would you look for?

3. For the geometry analysis (Module 3): which comparison is more interesting — 0-back vs 2-back (load effect) or target vs non-target within 2-back (detection effect)? Why?

*Write your answer to Exercise C in `notes/` as a design memo. This is real scientific reasoning.*

### Exercise D — The Epoch Window

We used pre=-200ms, post=1500ms. What happens if you change post to 3000ms? What about pre=-500ms? When would a longer baseline be better or worse?

---
## What You Now Have

- ✅ Clean, epoched high-gamma power: `epochs_z` with shape `(n_trials, n_times, n_channels)`
- ✅ A working preprocessing pipeline in `ieeg_utils.py`
- ✅ The basic WM load effect replicated (high-gamma scales with N-back level)
- ✅ A sense of which channels are informative
- ✅ A preview of the latent trajectory structure we'll learn in Module 2

## Next Steps

**Write** `notes/ecog_signal_preprocessing.md` — derive the CAR formula and explain why it removes common mode noise. Explain why Hilbert gives instantaneous power.

**Read** before Module 2:
- Kingma & Welling (2013) — VAE paper (8 pages)
- Pandarinath et al. (2018) LFADS — methods section

**Then:** `02_latent_trajectories/02_sequential_vae.ipynb`